
Statistical Tests for Prompt & Model Comparison
================================================
Analyzes argument quality scores (Likert 1–5) from multiple experts
to determine which prompt type (0-shot vs 1-shot) produces stronger arguments.

DESIGN: PAIRED samples — consecutive rows share the same input (opinions).
        Row i and row i+1 were generated from the same 3 opinions,
        one with 0-shot and one with 1-shot (order indicated by prompt_type).

Primary test: Wilcoxon signed-rank (non-parametric, paired).

Secondary:    Paired t-test (parametric reference).


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import (
    wilcoxon, shapiro, levene,
    ttest_rel, ttest_ind, mannwhitneyu,
    friedmanchisquare, kruskal
)
from itertools import combinations
import warnings
warnings.filterwarnings("ignore")

try:
    from statsmodels.stats.multitest import multipletests
    HAS_STATSMODELS = True
except ImportError:
    HAS_STATSMODELS = False
    print("⚠  statsmodels not installed — Bonferroni/BH correction will be skipped.")
    print("   pip install statsmodels\n")


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# 0. Dataset
#    Rows are consecutive pairs: row i and i+1 share the same input opinions.
#    prompt_type indicates which technique was used for each row.
# ─────────────────────────────────────────────────────────────────────────────

n_pairs = 18  # number of input pairs

df = pd.read_csv('experts_shot_results.csv')

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 36 entries, 0 to 35
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Unnamed: 0   36 non-null     int64 
 1   document_id  36 non-null     int64 
 2   expert1      36 non-null     int64 
 3   expert2      36 non-null     int64 
 4   expert3      36 non-null     int64 
 5   expert4      36 non-null     int64 
 6   gemma3       36 non-null     int64 
 7   0shot        36 non-null     int64 
 8   1shot        36 non-null     int64 
 9   shot         36 non-null     int64 
 10  prompt_type  36 non-null     object
dtypes: int64(10), object(1)
memory usage: 3.2+ KB


In [5]:
EXPERT_COLS  = ["expert1", "expert2", "expert3", "expert4"]
LIKERT_MIN, LIKERT_MAX = 1, 5

# ── Validate Likert range ────────────────────────────────────────────────────
invalid_mask = (df[EXPERT_COLS] < LIKERT_MIN) | (df[EXPERT_COLS] > LIKERT_MAX)
if invalid_mask.any().any():
    raise ValueError(f"⚠  Scores outside [{LIKERT_MIN}–{LIKERT_MAX}] detected!")
print(f"✅ All expert scores are valid integers in [{LIKERT_MIN}–{LIKERT_MAX}]\n")

# ─────────────────────────────────────────────────────────────────────────────
# BUILD PAIRED DATAFRAME
# Consecutive rows share the same input. We reconstruct pairs by iterating
# over the df in steps of 2 and assigning each row to its prompt_type.
# ─────────────────────────────────────────────────────────────────────────────
df["mean_score"]   = df[EXPERT_COLS].mean(axis=1)
df["median_score"] = df[EXPERT_COLS].median(axis=1)
df["score_std"]    = df[EXPERT_COLS].std(axis=1)

pairs = []
for i in range(0, len(df) - 1, 2):
    row_a = df.iloc[i]
    row_b = df.iloc[i + 1]
    # identify which is 0-shot and which is 1-shot
    if row_a["prompt_type"] == "0-shot" and row_b["prompt_type"] == "1-shot":
        zero, one = row_a, row_b
    elif row_a["prompt_type"] == "1-shot" and row_b["prompt_type"] == "0-shot":
        zero, one = row_b, row_a
    else:
        print(f"⚠  Pair at rows {i},{i+1} has duplicate prompt_type — skipped.")
        continue
    pairs.append({
        "pair_id":        i // 2 + 1,
        "doc_0shot":      zero["document_id"],
        "doc_1shot":      one["document_id"],
        "score_0shot":    zero["mean_score"],
        "score_1shot":    one["mean_score"],
        "diff":           zero["mean_score"] - one["mean_score"],   # positive = 0-shot better
    })
    for exp in EXPERT_COLS:
        pairs[-1][f"{exp}_0shot"] = zero[exp]
        pairs[-1][f"{exp}_1shot"] = one[exp]

paired_df = pd.DataFrame(pairs)
n_pairs_valid = len(paired_df)

# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────
SEP = "─" * 65

def section(title):
    print(f"\n{'═'*65}\n  {title}\n{'═'*65}")

def subsection(title):
    print(f"\n{SEP}\n  {title}\n{SEP}")

def sig_stars(p):
    if p < 0.001: return "***"
    if p < 0.01:  return "**"
    if p < 0.05:  return "*"
    return "ns"

def interpret(p, test=""):
    verdict = "✅ SIGNIFICANT" if p < 0.05 else "❌ not significant"
    return f"  → {test} p={p:.4f} {sig_stars(p)}  [{verdict}]"

# ─────────────────────────────────────────────────────────────────────────────
# 1. DESCRIPTIVE STATISTICS
# ─────────────────────────────────────────────────────────────────────────────
section("1. DESCRIPTIVE STATISTICS BY PROMPT TYPE")
desc = (
    df.groupby("prompt_type")["mean_score"]
    .agg(N="count", Mean="mean", Median="median",
         Std="std", Min="min", Max="max",
         Q25=lambda x: x.quantile(0.25),
         Q75=lambda x: x.quantile(0.75))
    .round(3)
)
print(desc.to_string())

subsection("1a. Paired differences (0-shot minus 1-shot per input)")
print(f"  N pairs:          {n_pairs_valid}")
print(f"  Mean difference:  {paired_df['diff'].mean():.4f}  (+ favors 0-shot)")
print(f"  Median difference:{paired_df['diff'].median():.4f}")
print(f"  Std of diffs:     {paired_df['diff'].std():.4f}")
print(f"  Pairs where 0-shot > 1-shot: {(paired_df['diff'] > 0).sum()}")
print(f"  Pairs where 1-shot > 0-shot: {(paired_df['diff'] < 0).sum()}")
print(f"  Ties:                        {(paired_df['diff'] == 0).sum()}")

# ─────────────────────────────────────────────────────────────────────────────
# 1b. SCORE DISTRIBUTION
# ─────────────────────────────────────────────────────────────────────────────
section("1b. LIKERT SCORE DISTRIBUTION (frequency %)")
for pt in ["0-shot", "1-shot"]:
    sub  = df.loc[df["prompt_type"] == pt, EXPERT_COLS].values.flatten()
    total = len(sub)
    counts = pd.Series(sub).value_counts().sort_index()
    print(f"  {pt}:")
    for score in range(LIKERT_MIN, LIKERT_MAX + 1):
        cnt = counts.get(score, 0)
        pct = 100 * cnt / total
        bar = "█" * int(pct / 2)
        print(f"    {score} │ {bar:<25} {cnt:>3} ({pct:4.1f}%)")
    mode_val = pd.Series(sub).mode()[0]
    print(f"    Mode: {mode_val}\n")

# ─────────────────────────────────────────────────────────────────────────────
# 1c. WEIGHTED KAPPA between expert pairs
# ─────────────────────────────────────────────────────────────────────────────
section("1c. PAIRWISE WEIGHTED KAPPA (quadratic weights, Likert 1-5)")

def weighted_kappa_quadratic(r1, r2, min_val=1, max_val=5):
    cats   = list(range(min_val, max_val + 1))
    n_cats = len(cats)
    n      = len(r1)
    W = np.array([[(i-j)**2 / (n_cats-1)**2 for j in cats] for i in cats])
    O = np.zeros((n_cats, n_cats))
    for a, b in zip(r1, r2):
        O[int(a)-min_val][int(b)-min_val] += 1
    O /= n
    E = np.outer(O.sum(axis=1), O.sum(axis=0))
    return 1 - (W*O).sum() / (W*E).sum() if (W*E).sum() != 0 else 1.0

kappa_results = []
for e1, e2 in combinations(EXPERT_COLS, 2):
    k = weighted_kappa_quadratic(df[e1].values, df[e2].values, LIKERT_MIN, LIKERT_MAX)
    kappa_results.append({"Pair": f"{e1} vs {e2}", "κ (quadratic)": round(k, 4)})

kappa_df = pd.DataFrame(kappa_results).set_index("Pair")
print(kappa_df.to_string())
mean_kappa = kappa_df["κ (quadratic)"].mean()
level_k = ("poor (<0.20)"            if mean_kappa < 0.20 else
           "fair (0.20–0.40)"        if mean_kappa < 0.40 else
           "moderate (0.40–0.60)"    if mean_kappa < 0.60 else
           "substantial (0.60–0.80)" if mean_kappa < 0.80 else
           "almost perfect (≥0.80)")
print(f"\n  Mean weighted κ = {mean_kappa:.4f}  ({level_k})")

# ─────────────────────────────────────────────────────────────────────────────
# 2. NORMALITY OF DIFFERENCES (key assumption for paired tests)
# ─────────────────────────────────────────────────────────────────────────────
section("2. NORMALITY OF PAIRED DIFFERENCES (Shapiro-Wilk)")
print("  ⚠  For paired tests, normality applies to the DIFFERENCES, not raw scores.\n")
sw_stat, sw_p = shapiro(paired_df["diff"])
verdict = "Normal ✅ → paired t-test also valid" if sw_p > 0.05 else "Non-normal ❌ → use Wilcoxon only"
print(f"  Differences: W={sw_stat:.4f}, p={sw_p:.4f}  → {verdict}")
diffs_normal = sw_p > 0.05

# ─────────────────────────────────────────────────────────────────────────────
# 3. PRIMARY PAIRED TESTS
# ─────────────────────────────────────────────────────────────────────────────
section("4. PRIMARY TESTS: PAIRED 0-SHOT vs 1-SHOT")

s0 = paired_df["score_0shot"].values
s1 = paired_df["score_1shot"].values
diffs = paired_df["diff"].values

subsection("4a. ★ Wilcoxon Signed-Rank Test (PRIMARY — non-parametric paired)")
print("  Correct test for paired ordinal/non-normal data.")
w_stat, w_p = wilcoxon(s0, s1, alternative="two-sided")
print(f"  W={w_stat:.4f}, p={w_p:.4f} {sig_stars(w_p)}")
print(interpret(w_p, "Wilcoxon"))

# Effect size r = Z / sqrt(N)
from scipy.stats import norm as sp_norm
z_score = sp_norm.ppf(w_p / 2)
r_wilcoxon = abs(z_score) / np.sqrt(n_pairs_valid)
mag_w = ("negligible" if r_wilcoxon < 0.1 else
         "small"      if r_wilcoxon < 0.3 else
         "medium"     if r_wilcoxon < 0.5 else "large")
print(f"  Effect size r = {r_wilcoxon:.4f}  ({mag_w})")

subsection("4b. Paired t-test (parametric reference)")
t_stat, t_p = ttest_rel(s0, s1)
print(f"  t={t_stat:.4f}, p={t_p:.4f} {sig_stars(t_p)}")
print(interpret(t_p, "paired t-test"))

# Cohen's d for paired (using std of differences)
cohens_dz = diffs.mean() / diffs.std(ddof=1) if diffs.std(ddof=1) > 0 else 0
mag_d = ("negligible" if abs(cohens_dz) < 0.2 else
         "small"      if abs(cohens_dz) < 0.5 else
         "medium"     if abs(cohens_dz) < 0.8 else "large")
print(f"\n  Cohen's dz = {cohens_dz:.4f}  ({mag_d} effect)")

subsection("4c. Mann-Whitney U (shown for reference — LESS appropriate for paired data)")
u_stat, u_p = mannwhitneyu(s0, s1, alternative="two-sided")
print(f"  U={u_stat:.4f}, p={u_p:.4f} {sig_stars(u_p)}")
print(interpret(u_p, "Mann-Whitney U"))

# ─────────────────────────────────────────────────────────────────────────────
# 5. PER-EXPERT PAIRED TESTS (Wilcoxon per expert)
# ─────────────────────────────────────────────────────────────────────────────
section("5. PER-EXPERT WILCOXON TESTS (paired)")
expert_results = []
for exp in EXPERT_COLS:
    e0 = paired_df[f"{exp}_0shot"].values
    e1 = paired_df[f"{exp}_1shot"].values
    try:
        w, p = wilcoxon(e0, e1, alternative="two-sided")
    except ValueError:
        w, p = np.nan, 1.0   # all differences are zero
    expert_results.append({
        "Expert": exp, "W": w, "p-value": p, "sig": sig_stars(p),
        "Mean 0-shot": round(e0.mean(), 3),
        "Mean 1-shot": round(e1.mean(), 3),
        "Mean diff":   round((e0 - e1).mean(), 3),
    })

exp_df = pd.DataFrame(expert_results).set_index("Expert")
print(exp_df.to_string())

if HAS_STATSMODELS:
    subsection("5a. Multiple Comparisons Correction")
    raw_ps = exp_df["p-value"].values
    for method in ["bonferroni", "fdr_bh"]:
        reject, p_corr, _, _ = multipletests(raw_ps, method=method)
        label = "Bonferroni" if method == "bonferroni" else "Benjamini-Hochberg (FDR)"
        print(f"\n  {label}:")
        for exp, pc, rej in zip(EXPERT_COLS, p_corr, reject):
            print(f"    {exp}: p_adj={pc:.4f}  {'✅ sig' if rej else '❌ ns'}")

# ─────────────────────────────────────────────────────────────────────────────
# 6. INTER-RATER RELIABILITY
# ─────────────────────────────────────────────────────────────────────────────
section("6. INTER-RATER RELIABILITY")

def krippendorffs_alpha(data):
    data = np.array(data, dtype=float)
    n_raters, n_items = data.shape
    Do = 0.0
    for i in range(n_items):
        vals = data[:, i][~np.isnan(data[:, i])]
        m = len(vals)
        if m < 2: continue
        Do += np.sum([(v1-v2)**2 for j,v1 in enumerate(vals) for v2 in vals[j+1:]]) / (m-1)
    all_vals = data[~np.isnan(data)]
    n = len(all_vals)
    De = np.sum([(v1-v2)**2 for j,v1 in enumerate(all_vals) for v2 in all_vals[j+1:]]) / (n-1)
    return 1 - Do/De if De != 0 else 1.0

alpha = krippendorffs_alpha(df[EXPERT_COLS].values.T)
level_a = ("poor (<0.20)"            if alpha < 0.20 else
           "fair (0.20–0.40)"        if alpha < 0.40 else
           "moderate (0.40–0.60)"    if alpha < 0.60 else
           "substantial (0.60–0.80)" if alpha < 0.80 else
           "almost perfect (≥0.80)")
print(f"  Krippendorff's α = {alpha:.4f}  ({level_a})")

subsection("6a. Intraclass Correlation (ICC 2,1)")
n_subj     = len(df)
n_rat      = len(EXPERT_COLS)
grand_mean = df[EXPERT_COLS].values.mean()
ss_subj    = n_rat * ((df[EXPERT_COLS].mean(axis=1) - grand_mean)**2).sum()
ss_raters  = n_subj * ((df[EXPERT_COLS].mean(axis=0) - grand_mean)**2).sum()
ss_total   = ((df[EXPERT_COLS].values - grand_mean)**2).sum()
ss_error   = ss_total - ss_subj - ss_raters
ms_subj    = ss_subj / (n_subj - 1)
ms_error   = ss_error / ((n_subj - 1) * (n_rat - 1))
icc        = (ms_subj - ms_error) / (ms_subj + (n_rat - 1) * ms_error)
print(f"  ICC(2,1) = {icc:.4f}  ({'good' if icc > 0.75 else 'moderate' if icc > 0.5 else 'poor'})")

subsection("6b. Friedman Test (consistency across experts per prompt type)")
for pt in ["0-shot", "1-shot"]:
    sub = df.loc[df["prompt_type"] == pt, EXPERT_COLS]
    f_stat, f_p = friedmanchisquare(*[sub[c] for c in EXPERT_COLS])
    print(f"  {pt}: χ²={f_stat:.4f}, p={f_p:.4f} {sig_stars(f_p)}")

# ─────────────────────────────────────────────────────────────────────────────
# 7. BOOTSTRAP CI ON PAIRED DIFFERENCES
# ─────────────────────────────────────────────────────────────────────────────
section("7. BOOTSTRAP 95% CI ON PAIRED DIFFERENCES (n=10 000 resamp.)")

def bootstrap_ci(data, n_boot=10_000, ci=0.95):
    means = [np.random.choice(data, size=len(data), replace=True).mean() for _ in range(n_boot)]
    lo = np.percentile(means, (1-ci)/2*100)
    hi = np.percentile(means, (1+ci)/2*100)
    return lo, hi

lo, hi = bootstrap_ci(diffs)
print(f"  Mean diff (0-shot − 1-shot): {diffs.mean():.4f}")
print(f"  95% CI: [{lo:.4f}, {hi:.4f}]")
if lo > 0:
    print("  → Entire CI above 0: 0-shot consistently better ✅")
elif hi < 0:
    print("  → Entire CI below 0: 1-shot consistently better ✅")
else:
    print("  → CI crosses 0: difference not conclusive ⚠")

for pt in ["0-shot", "1-shot"]:
    scores = df.loc[df["prompt_type"] == pt, "mean_score"].values
    lo_s, hi_s = bootstrap_ci(scores)
    print(f"\n  {pt}: mean={scores.mean():.3f}  95% CI [{lo_s:.3f}, {hi_s:.3f}]")

# ─────────────────────────────────────────────────────────────────────────────
# 8. SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
section("8. SUMMARY: WHICH PROMPT TYPE IS STRONGER?")

winner = "0-shot" if s0.mean() > s1.mean() else "1-shot"
print(f"  Mean scores    →  0-shot: {s0.mean():.3f}  |  1-shot: {s1.mean():.3f}")
print(f"  Mean diff      →  Δ = {diffs.mean():.3f}  (favors {winner})")
print(f"  ★ Wilcoxon     →  p={w_p:.4f} {sig_stars(w_p)}  ← PRIMARY test")
print(f"  Paired t-test  →  p={t_p:.4f} {sig_stars(t_p)}")
print(f"  Cohen's dz     →  {cohens_dz:.4f}  ({mag_d} effect)")
print(f"  Wilcoxon r     →  {r_wilcoxon:.4f}  ({mag_w} effect)")
print(f"  ICC            →  {icc:.4f}")
print(f"  Bootstrap CI   →  [{lo:.4f}, {hi:.4f}]  {'✅ excludes 0' if lo > 0 or hi < 0 else '⚠ includes 0'}")

primary_p = w_p
if primary_p < 0.05:
    print(f"\n  🏆 CONCLUSION: '{winner}' produces significantly stronger arguments.")
else:
    print(f"\n  ⚖  CONCLUSION: Trend favors {winner} but not statistically significant.")
    print(f"     Consider increasing sample size (current n_pairs={n_pairs_valid}).")

print(f"\n  Significance legend:  *** p<0.001  ** p<0.01  * p<0.05  ns = not significant")
print()

✅ All expert scores are valid integers in [1–5]


═════════════════════════════════════════════════════════════════
  1. DESCRIPTIVE STATISTICS BY PROMPT TYPE
═════════════════════════════════════════════════════════════════
              N   Mean  Median    Std   Min   Max    Q25    Q75
prompt_type                                                    
0-shot       18  3.889   4.250  0.850  2.25  4.75  3.500  4.688
1-shot       18  3.333   3.625  1.022  1.25  4.75  2.562  4.000

─────────────────────────────────────────────────────────────────
  1a. Paired differences (0-shot minus 1-shot per input)
─────────────────────────────────────────────────────────────────
  N pairs:          18
  Mean difference:  0.5556  (+ favors 0-shot)
  Median difference:0.8750
  Std of diffs:     1.5801
  Pairs where 0-shot > 1-shot: 12
  Pairs where 1-shot > 0-shot: 6
  Ties:                        0

═════════════════════════════════════════════════════════════════
  1b. LIKERT SCORE DISTRIBUTION (freque